In [0]:
-- Business Question 3: Zone-Level Mobility Patterns & Opportunities
-- Grain: Day of week + hour + zone role + taxi zone + weather condition
--
-- Pickup activity uses pickup date/hour.
-- Dropoff activity uses dropoff date/hour.
--
-- Weather is recorded at pickup time in fact_trip. Therefore, weather
-- shown for dropoff activity represents the pickup weather of those trips.
--
-- Fare, distance, and duration metrics are attributed to both pickup
-- and dropoff zone activity. Do not sum these metrics across all zones
-- to obtain systemwide totals.

WITH pickup_activity AS (
    SELECT
        d.day_of_week,
        d.day_name,
        h.hour_of_day AS hour,
        h.day_period AS time_of_day,

        w.temperature_band,
        w.weather_condition,
        w.is_raining AS precipitation_flag,

        'pickup' AS zone_role,
        t.pickup_zone_key AS zone_key,

        SUM(t.trip_count) AS trip_count,
        SUM(t.trip_count) AS pickups,
        0 AS dropoffs,

        COUNT(t.fare_amount) AS fare_count,
        SUM(t.fare_amount) AS total_fare,

        COUNT(t.trip_distance) AS distance_count,
        SUM(t.trip_distance) AS total_distance,

        COUNT(t.trip_duration_minutes) AS duration_count,
        SUM(t.trip_duration_minutes) AS total_duration_minutes

    FROM nyc_mobility.gold.fact_trip AS t

    JOIN nyc_mobility.gold.dim_date AS d
        ON t.pickup_date_key = d.date_key

    JOIN nyc_mobility.gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key

    JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key

    GROUP BY
        d.day_of_week,
        d.day_name,
        h.hour_of_day,
        h.day_period,
        w.temperature_band,
        w.weather_condition,
        w.is_raining,
        t.pickup_zone_key
),

dropoff_activity AS (
    SELECT
        d.day_of_week,
        d.day_name,
        h.hour_of_day AS hour,
        h.day_period AS time_of_day,

        w.temperature_band,
        w.weather_condition,
        w.is_raining AS precipitation_flag,

        'dropoff' AS zone_role,
        t.dropoff_zone_key AS zone_key,

        SUM(t.trip_count) AS trip_count,
        0 AS pickups,
        SUM(t.trip_count) AS dropoffs,

        COUNT(t.fare_amount) AS fare_count,
        SUM(t.fare_amount) AS total_fare,

        COUNT(t.trip_distance) AS distance_count,
        SUM(t.trip_distance) AS total_distance,

        COUNT(t.trip_duration_minutes) AS duration_count,
        SUM(t.trip_duration_minutes) AS total_duration_minutes

    FROM nyc_mobility.gold.fact_trip AS t

    JOIN nyc_mobility.gold.dim_date AS d
        ON t.dropoff_date_key = d.date_key

    JOIN nyc_mobility.gold.dim_hour AS h
        ON t.dropoff_hour_key = h.hour_key

    JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key

    GROUP BY
        d.day_of_week,
        d.day_name,
        h.hour_of_day,
        h.day_period,
        w.temperature_band,
        w.weather_condition,
        w.is_raining,
        t.dropoff_zone_key
),

zone_activity AS (
    SELECT * FROM pickup_activity

    UNION ALL

    SELECT * FROM dropoff_activity
),

aggregated AS (
    SELECT
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.weather_condition,
        a.precipitation_flag,
        a.zone_role,
        a.zone_key,

        z.borough,
        z.zone,

        SUM(a.trip_count) AS trip_count,
        SUM(a.pickups) AS pickups,
        SUM(a.dropoffs) AS dropoffs,

        SUM(a.fare_count) AS fare_count,
        SUM(a.total_fare) AS total_fare,

        SUM(a.distance_count) AS distance_count,
        SUM(a.total_distance) AS total_distance,

        SUM(a.duration_count) AS duration_count,
        SUM(a.total_duration_minutes) AS total_duration_minutes

    FROM zone_activity AS a

    JOIN nyc_mobility.gold.dim_zone AS z
        ON a.zone_key = z.zone_key

    GROUP BY
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.weather_condition,
        a.precipitation_flag,
        a.zone_role,
        a.zone_key,
        z.borough,
        z.zone
),

zone_flow AS (
    SELECT
        day_of_week,
        day_name,
        hour,
        time_of_day,
        temperature_band,
        weather_condition,
        precipitation_flag,
        zone_key,

        SUM(pickups) AS pickups,
        SUM(dropoffs) AS dropoffs,

        SUM(dropoffs) - SUM(pickups) AS net_flow,

        ABS(
            SUM(dropoffs) - SUM(pickups)
        ) AS absolute_flow_imbalance

    FROM aggregated

    GROUP BY
        day_of_week,
        day_name,
        hour,
        time_of_day,
        temperature_band,
        weather_condition,
        precipitation_flag,
        zone_key
)

SELECT
    a.day_of_week,
    a.day_name,
    a.hour,
    a.time_of_day,

    a.temperature_band,
    a.weather_condition,
    a.precipitation_flag,

    a.zone_role,
    a.zone_key,
    a.borough,
    a.zone,

    a.trip_count,

    f.pickups,
    f.dropoffs,
    f.net_flow,
    f.absolute_flow_imbalance,

    f.pickups + f.dropoffs AS total_zone_activity,

    a.fare_count,
    a.total_fare,

    a.distance_count,
    a.total_distance,

    a.duration_count,
    a.total_duration_minutes,

    ROUND(
        a.total_distance
        / NULLIF(a.distance_count, 0),
        2
    ) AS avg_trip_distance,

    ROUND(
        SUM(a.total_distance)
            OVER (PARTITION BY a.zone_role, a.borough)
        / NULLIF(
            SUM(a.distance_count)
                OVER (PARTITION BY a.zone_role, a.borough),
            0
        ),
        2
    ) AS avg_trip_distance_by_borough,

    ROUND(
        SUM(a.total_fare)
            OVER (PARTITION BY a.zone_role, a.borough)
        / NULLIF(
            SUM(a.fare_count)
                OVER (PARTITION BY a.zone_role, a.borough),
            0
        ),
        2
    ) AS avg_fare_by_borough,

    ROUND(
        a.total_fare
        / NULLIF(a.fare_count, 0),
        2
    ) AS avg_fare,

    ROUND(
        a.total_duration_minutes
        / NULLIF(a.duration_count, 0),
        2
    ) AS avg_trip_duration_minutes

FROM aggregated AS a

JOIN zone_flow AS f
    ON a.day_of_week = f.day_of_week
    AND a.hour = f.hour
    AND a.temperature_band = f.temperature_band
    AND a.weather_condition = f.weather_condition
    AND a.precipitation_flag = f.precipitation_flag
    AND a.zone_key = f.zone_key

ORDER BY
    f.absolute_flow_imbalance DESC,
    a.trip_count DESC;